# 時系列データの前処理と再帰予測（リークなし）: date・Y_cat・Y・X1〜X50

- **作成者**: ImoZukushi
- **作成日**: 2026-10-21（2026-10-25 long 型に変更、欠損のあるデータに対応）
- **目的**: `data/` 以下の long 型の学習データ（列 `date`, `Y_cat`, `Y`, `X1`〜`X50`）とテストデータ
  （列 `date`, `Y_cat`, `X1`〜`X50`）を読み込み、目的変数・日付・説明変数のそれぞれに `src/feature_engineering` の
  前処理をかけて CSV に出力する。目的変数のラグ・移動平均は、テスト期間では**再帰予測の中で**（予測値を履歴に
  足しながら）作る。目的変数は種類（`Y_cat` = Y1〜Y10）ごとに別の系列で、ここでは種類ごとに別のモデルで予測する。
- **データの形（long 型）**: 1行 = 日付 × 目的変数の種類。目的変数の値は `Y`、種類は `Y_cat`。説明変数は日付ごとの値なので、
  同じ日付の行では同じ値になる。
- **欠損**: 目的変数・説明変数の両方に、散発的な欠損と連続した欠損がある（学習・テストの両方）。
  欠損は埋めず、欠損のまま扱う（「リークを防ぐための決まり」の 6）。
- **入力**（ダミー。`scripts/generate_dummy_x50.py` で作成）
  - 学習: `data/synthetic/x50/dummy_x50_train.csv`（2022-01-01〜2024-12-31）
  - テスト: `data/synthetic/x50/dummy_x50_test.csv`（2025-01-01〜2025-12-31、目的変数の値なし）
  - 正解: `data/synthetic/x50/dummy_x50_test_answer.csv`（列 `date`, `Y_cat`, `Y`。予測の評価だけに使う）
- **出力**（`data/processed/x50/`。いずれも long 型）
  - `x50_train_features.csv`: 学習データの前処理結果（種類ごとに作った目的変数のラグ・移動平均を含む）
  - `x50_test_features.csv`: テストデータの前処理結果（目的変数の特徴量は含まない）
  - `x50_test_predictions.csv`: 種類ごとの再帰予測の結果と、その予測に使った目的変数の特徴量

## リークを防ぐための決まり

1. **値を学習する前処理は、学習データだけで fit する。**
   べき変換の λ、スケーリングの中央値・四分位範囲、経過日数の基準日、どの列をべき変換するかの判定は、
   学習データだけで決める。テストデータには `transform` だけをかける。
2. **テストデータの目的変数は使わない。**
   テストデータには目的変数の値（`Y`）の列が無い（あれば読み込み時に外す）。正解ファイルは、最後の評価でだけ読む。
3. **目的変数のラグ・移動平均は、予測時点で分かっている値だけから作る。**
   - ラグ等は種類（`Y_cat`）ごとに作る（`Y_cat` = Y1 の行のラグは、Y1 の過去の値だけから作る）。
   - 学習データでは、各日 t の特徴量を t − 1 日以前の実測値から作る（当日の値を含まない）。
   - テスト期間では実測値が分からないため、特徴量を事前に作らない。再帰予測で1日ずつ進み、
     「学習期間の実測値 + それまでに予測した値」から、その日の特徴量を作ってから予測する。
4. **種類ごとのモデルは、その種類の特徴量だけを使う。**
   Yk のモデルに他の種類（Yj）のラグを入れると、テスト期間では Yj も予測しないと値が作れない。
   そのため Yk のモデルは `Y_cat` = Yk の行だけで、「自身のラグ・移動平均 + 日付 + 説明変数」から学習する。
5. **説明変数は、テスト期間の値が予測時点で分かっているとみなす**（テストデータに与えられているため）。
6. **欠損は、未来の値で埋めない。** 前後の値で補間すると、学習データの中でもその日より後の値を使うことになる。
   そのため欠損は埋めず、次のように扱う。
   - 目的変数が欠損の行は、モデルの学習（目的変数として）と評価から除く。行は消さずに残し、ラグ・移動平均は
     欠損のまま作る（移動平均は窓内の観測値だけで平均する。`ROLLING_MIN_PERIODS`）。
   - 説明変数・目的変数の特徴量の欠損は、そのままモデル（LightGBM / XGBoost の欠損の扱い）に任せる。
   - テストの正解が欠損の日は、評価（MAE）から除く。

**前提:** 種類ごとに 1行 = 1日で、日付に欠けや重複がないこと。テストデータは学習データの最終日の翌日から始まること。
読み込み時に確認する。

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# pyproject.toml を目印にリポジトリルートを探し、src/ をimportパスに追加する
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# src/ 配下のモジュール（上でパスを追加した後にimportする必要がある）
from feature_engineering.datetime_features import (  # noqa: E402
    CyclicalFeaturesEncoder,
    DatetimeFeaturesExtractor,
    ElapsedTimeTransformer,
)
from feature_engineering.numeric import LogTransformer, PandasPowerTransformer  # noqa: E402
from feature_engineering.scaling import PandasRobustScaler  # noqa: E402
from modeling.config import ForecastConfig  # noqa: E402
from modeling.forecasting import TargetFeatureBuilder, recursive_forecast  # noqa: E402
from util.csv_io import read_csv_auto  # noqa: E402
from util.paths import data_dir, ensure_parent_dir  # noqa: E402

# pandas の表示で列を省略しすぎないようにする
pd.set_option("display.max_columns", 30)


def shown(path: Path) -> str:
    """表示用のパス（リポジトリ内ならルートからの相対パス）。"""
    resolved = path.resolve()
    if resolved.is_relative_to(REPO_ROOT):
        return resolved.relative_to(REPO_ROOT).as_posix()
    return str(path)

## パラメータ

papermill で実行するときは `-p 名前 値` で上書きできる（このセルに `parameters` タグを付けている）。
リストや真偽値は `-y` で YAML として渡す。

```bash
uv run papermill notebook/003_feature_engineering_x50.ipynb out.ipynb --cwd notebook \
    -y "{TARGET_CATS: [Y1, Y2], TARGET_LAGS: [1, 7, 14], RATE_OF_CHANGE: false}"
```

In [2]:
# 入力・出力（data/ からの相対パス）。ANSWER_PATH を None にすると評価を飛ばす
TRAIN_PATH = "synthetic/x50/dummy_x50_train.csv"
TEST_PATH = "synthetic/x50/dummy_x50_test.csv"
ANSWER_PATH = "synthetic/x50/dummy_x50_test_answer.csv"
OUTPUT_DIR = "processed/x50"

# 列の指定（long 型）
DATE_COL = "date"  # 日付
CAT_COL = "Y_cat"  # 目的変数の種類（Y1〜Y10。種類ごとに別の系列）
TARGET_COL = "Y"  # 目的変数の値
TARGET_CATS = None  # 予測する種類（None なら学習データの全種類。データに出てくる順）
FEATURE_COLS = None  # 説明変数（None なら日付・種類・目的変数以外の全列）

# 目的変数の特徴量（全種類で同じ定義。種類ごとに作り、再帰予測でも同じ定義で作る）
TARGET_LAGS = [1, 2, 3, 7, 14, 28]  # ラグ（日数。1以上）
ROLLING_WINDOWS = [7, 28]  # 前日までの移動平均の窓幅（日数）
# 移動平均に必要な、窓内の観測値（欠損でない値）の数。
# 目的変数に欠損があっても、窓内の観測値だけで平均する
ROLLING_MIN_PERIODS = 1
RATE_OF_CHANGE = True  # 前日の前日比を加えるか
# 目的変数の変換（"log" または None）。モデルは変換後の値を学習・予測する
TARGET_TRANSFORM = "log"
TARGET_LOG_OFFSET = 0.0

# 日付から取り出す成分
DATE_COMPONENTS = ["year", "month", "day", "weekday", "week"]
# 学習データの歪度の絶対値がこれを超える説明変数は、べき変換（Yeo-Johnson）で分布を整える
SKEW_THRESHOLD = 1.0

# 再帰予測に使うモデル（LightGBM）のパラメータ（全種類で共通）
MODEL_PARAMS = {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 15, "random_state": 0}

In [3]:
train_path = data_dir() / TRAIN_PATH
test_path = data_dir() / TEST_PATH
answer_path = None if ANSWER_PATH is None else data_dir() / ANSWER_PATH
output_dir = data_dir() / OUTPUT_DIR
# 出力先が生データ（data/raw・data/external）でないことを確かめる（生データは書き換えない）
for immutable in ("raw", "external"):
    if output_dir.resolve().is_relative_to((data_dir() / immutable).resolve()):
        raise ValueError(f"data/{immutable} には書き込めません: {output_dir}")

# 目的変数の特徴量の定義（学習データ・再帰予測の両方で同じものを使う）。
# 種類（Y_cat）ごとに作る。ラグは1日以上前、移動平均・前日比は前日のラグ列から作るため、
# 当日の値を含まない
builder = TargetFeatureBuilder(
    TARGET_COL,
    DATE_COL,
    ForecastConfig(
        series_col=CAT_COL,
        lags=TARGET_LAGS,
        rolling_windows=ROLLING_WINDOWS,
        rolling_min_periods=ROLLING_MIN_PERIODS,
        rate_of_change=RATE_OF_CHANGE,
    ),
)
target_feature_cols = builder.feature_names

print(f"学習: {shown(train_path)}")
print(f"テスト: {shown(test_path)}")
print(f"出力: {shown(output_dir)}")
print(f"目的変数の特徴量（種類ごとに作る）: {target_feature_cols}")
print(f"再帰予測で必要な過去の日数: {builder.lookback}")

学習: data/synthetic/x50/dummy_x50_train.csv
テスト: data/synthetic/x50/dummy_x50_test.csv
出力: data/processed/x50
目的変数の特徴量（種類ごとに作る）: ['Y_lag_1', 'Y_lag_2', 'Y_lag_3', 'Y_lag_7', 'Y_lag_14', 'Y_lag_28', 'Y_lag_1_ma_7', 'Y_lag_1_ma_28', 'Y_lag_1_roc_1']
再帰予測で必要な過去の日数: 28


## 1. 読み込みと確認

`util.csv_io.read_csv_auto` で読み込む（文字コードの判定・日付列の変換を自動で行う）。
行を「日付の昇順、同じ日付の中は種類の順」に並べ、種類ごとに 1行 = 1日か（重複・欠けが無いか）、
テストデータが学習データの翌日から続いているかを確かめる。目的変数・説明変数の欠損の状況（割合と、最も長い連続した
欠損の日数）も確かめる。

In [4]:
def sort_long(frame: pd.DataFrame, cats: list[str]) -> pd.DataFrame:
    """long 型の表を、日付の昇順・同じ日付の中は種類の順（cats の順）に並べる。"""
    order = {cat: i for i, cat in enumerate(cats)}
    return frame.sort_values(
        [DATE_COL, CAT_COL],
        key=lambda s: s.map(order) if s.name == CAT_COL else s,
        kind="stable",
    ).reset_index(drop=True)


def check_daily_by_category(frame: pd.DataFrame, name: str) -> None:
    """種類ごとに、日付が1日刻みで重複も欠けもないことを確かめる。"""
    for cat, part in frame.groupby(CAT_COL, sort=False):
        dates = pd.to_datetime(part[DATE_COL])
        if dates.duplicated().any():
            raise ValueError(f"{name}（{cat}）: 日付が重複しています")
        if not (dates.diff().dropna() == pd.Timedelta(days=1)).all():
            raise ValueError(
                f"{name}（{cat}）: 日付に欠けがあります（1行=1日の前提で特徴量を作るため）"
            )


train_raw = read_csv_auto(train_path)
test_raw = read_csv_auto(test_path)
for name, frame, required in (
    ("学習データ", train_raw, [DATE_COL, CAT_COL, TARGET_COL]),
    ("テストデータ", test_raw, [DATE_COL, CAT_COL]),
):
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f"{name}に列がありません: {missing}")
if TARGET_COL in test_raw.columns:
    # テスト期間の目的変数の値は予測時点で分からないので、あっても使わない
    print(f"テストデータの {TARGET_COL} 列は使わないため外します")
    test_raw = test_raw.drop(columns=TARGET_COL)

# 予測する種類（データに出てくる順）
cats = TARGET_CATS or list(dict.fromkeys(train_raw[CAT_COL]))
missing_cats = [
    c for c in cats if c not in set(train_raw[CAT_COL]) or c not in set(test_raw[CAT_COL])
]
if missing_cats:
    raise ValueError(f"学習データ・テストデータの両方にない種類があります: {missing_cats}")

excluded = {DATE_COL, CAT_COL, TARGET_COL}
feature_cols = FEATURE_COLS or [c for c in train_raw.columns if c not in excluded]
missing_in_test = [c for c in feature_cols if c not in test_raw.columns]
if missing_in_test:
    raise ValueError(f"テストデータに説明変数がありません: {missing_in_test}")
train_raw = sort_long(
    train_raw.loc[train_raw[CAT_COL].isin(cats), [DATE_COL, CAT_COL, TARGET_COL, *feature_cols]],
    cats,
)
test_raw = sort_long(
    test_raw.loc[test_raw[CAT_COL].isin(cats), [DATE_COL, CAT_COL, *feature_cols]], cats
)

check_daily_by_category(train_raw, "学習データ")
check_daily_by_category(test_raw, "テストデータ")
train_end = train_raw.groupby(CAT_COL)[DATE_COL].max()
test_start = test_raw.groupby(CAT_COL)[DATE_COL].min()
not_next = [c for c in cats if test_start[c] != train_end[c] + pd.Timedelta(days=1)]
if not_next:
    raise ValueError(f"テストデータが学習データの翌日から始まっていない種類があります: {not_next}")

for name, frame in (("学習", train_raw), ("テスト", test_raw)):
    dates = frame[DATE_COL]
    n_missing_cols = int((frame.isna().sum() > 0).sum())
    n_days, n_cats = dates.nunique(), frame[CAT_COL].nunique()
    print(
        f"{name}: {len(frame)}行 = {n_days}日 × {n_cats}種類, {frame.shape[1]}列"
        f"（{dates.min():%Y-%m-%d}〜{dates.max():%Y-%m-%d}） 欠損のある列: {n_missing_cols}"
    )
print(f"種類: {cats}")


def longest_run(missing: pd.Series) -> int:
    """欠損（True）が連続する最長の日数。"""
    runs = missing.ne(missing.shift()).cumsum()
    return int(missing.groupby(runs).sum().max()) if missing.any() else 0


# 目的変数の欠損（種類ごと）。説明変数は日付ごとの値なので、1つの種類の行で数える
target_by_cat = train_raw.groupby(CAT_COL, sort=False)[TARGET_COL]
target_missing = pd.DataFrame(
    {
        "学習の欠損率": target_by_cat.apply(lambda s: s.isna().mean()),
        "最長の連続欠損の日数": target_by_cat.apply(lambda s: longest_run(s.isna())),
    }
)
one_cat = train_raw[train_raw[CAT_COL] == cats[0]]
one_cat_test = test_raw[test_raw[CAT_COL] == cats[0]]
feature_missing = pd.DataFrame(
    {
        "学習の欠損率": one_cat[feature_cols].isna().mean(),
        "テストの欠損率": one_cat_test[feature_cols].isna().mean(),
        "学習の最長の連続欠損の日数": [longest_run(one_cat[c].isna()) for c in feature_cols],
    }
)
print("目的変数の欠損（学習データ。種類ごと）:")
display(target_missing.round(3).T)
print("説明変数の欠損（欠損率の大きい順に10列）:")
display(feature_missing.sort_values("学習の欠損率", ascending=False).head(10).round(3))
# 種類ごとの水準・ばらつき（尺度が大きく違うので、MAE は種類ごとに比べる）
train_raw.groupby(CAT_COL, sort=False)[TARGET_COL].describe()[["mean", "std", "min", "max"]].round(
    2
)

学習: 10960行 = 1096日 × 10種類, 53列（2022-01-01〜2024-12-31） 欠損のある列: 51
テスト: 3650行 = 365日 × 10種類, 52列（2025-01-01〜2025-12-31） 欠損のある列: 50
種類: ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6', 'Y7', 'Y8', 'Y9', 'Y10']
目的変数の欠損（学習データ。種類ごと）:


Y_cat,Y1,Y2,Y3,Y4,Y5,Y6,Y7,Y8,Y9,Y10
学習の欠損率,0.047,0.055,0.058,0.054,0.047,0.063,0.043,0.053,0.063,0.064
最長の連続欠損の日数,15.000,28.000,22.000,32.000,21.000,34.000,31.000,33.000,26.000,30.000


説明変数の欠損（欠損率の大きい順に10列）:


,学習の欠損率,テストの欠損率,学習の最長の連続欠損の日数
X24,0.057,0.099,27
X42,0.054,0.093,28
X8,0.050,0.011,30
X2,0.048,0.068,28
X7,0.048,0.008,26
X4,0.047,0.052,26
X48,0.047,0.036,28
X25,0.045,0.022,5
X13,0.045,0.055,31
X50,0.043,0.005,30


,mean,std,min,max
Y_cat,,,,
Y1,59.78,13.43,21.32,94.80
Y2,86.32,14.02,46.13,132.19
Y3,79.48,12.23,45.05,108.75
Y4,30.08,3.66,19.39,40.90
Y5,77.87,12.29,46.62,123.19
Y6,150.09,29.66,72.83,223.38
Y7,174.69,21.36,111.80,232.21
Y8,187.71,36.23,98.79,277.43
Y9,139.86,25.05,62.79,214.24


## 2. 前処理の定義

各変数への前処理は次のとおり。学習が必要なもの（★）は学習データだけで fit する。

| 対象 | 前処理 | 作られる列 | 学習データ | テストデータ |
|---|---|---|---|---|
| 目的変数 | `feature_engineering.numeric.LogTransformer`（`TARGET_TRANSFORM="log"`） | `Y`（変換後。元の値は `Y_raw`） | ○ | （目的変数なし） |
| 目的変数 | `modeling.forecasting.TargetFeatureBuilder`（`Y_cat` ごと。内部で `feature_engineering.time_series` のラグ・移動平均・変化率） | `Y_lag_{n}`, `Y_lag_1_ma_{w}`, `Y_lag_1_roc_1` | ○ | **再帰予測の中で作る** |
| 日付 | `datetime_features.DatetimeFeaturesExtractor` | `date_year`, `date_month`, ... | ○ | ○ |
| 日付 | `datetime_features.CyclicalFeaturesEncoder`（月=12, 曜日=7） | `date_month_sin/cos`, `date_weekday_sin/cos` | ○ | ○ |
| 日付 | ★ `datetime_features.ElapsedTimeTransformer`（基準日 = 学習データの初日） | `date_elapsed_days` | ○ | ○ |
| 説明変数 | ★ `numeric.PandasPowerTransformer`（学習データで歪度が大きい列だけ） | 同じ列名で置き換え | ○ | ○ |
| 説明変数 | ★ `scaling.PandasRobustScaler` | 同じ列名で置き換え | ○ | ○ |

欠損はどの前処理でも埋めない（欠損のまま変換し、`Y` が欠損なら `Y_raw` も欠損）。べき変換・スケーリングの値は、
学習データの欠損でない値から求める。目的変数のラグ・移動平均は欠損のまま作り、移動平均は窓内の観測値だけで平均する
（`ROLLING_MIN_PERIODS`）。

学習データで fit した前処理は `Preprocessor` にまとめ、テストデータには同じものの `transform` だけをかける。
long 型では説明変数の値が種類の数だけ繰り返されるが、全ての日付が同じ回数ずつ繰り返されるため、
学習する値（分位数・歪度・λ）は日付ごとの値で計算した場合と同じになる。

In [5]:
class Preprocessor:
    """目的変数の変換・日付の特徴量・説明変数の変換をまとめた前処理（学習データだけで fit する）。

    目的変数のラグ・移動平均は含まない
    （学習データでは `TargetFeatureBuilder.build`、テスト期間では再帰予測で作る）。

    Attributes:
        target_transformer: 目的変数の変換（変換しないならNone）。
        skewed: べき変換する説明変数（学習データの歪度で判定）。
        stateful: fit済みの、学習が必要な前処理（経過日数・べき変換・スケーリング）。
    """

    def fit(self, train: pd.DataFrame) -> "Preprocessor":
        """学習データで、変換の対象と値を決める。"""
        self.target_transformer = None
        if TARGET_TRANSFORM == "log":
            self.target_transformer = LogTransformer(TARGET_COL, offset=TARGET_LOG_OFFSET).fit(
                train
            )
        elif TARGET_TRANSFORM is not None:
            raise ValueError(f"未対応の TARGET_TRANSFORM です: {TARGET_TRANSFORM}")
        dated = self._date_features(train)
        self.skewed = [c for c in feature_cols if abs(dated[c].skew()) > SKEW_THRESHOLD]
        steps: list[tuple[str, object]] = [
            ("elapsed", ElapsedTimeTransformer(DATE_COL, unit="days"))
        ]
        if self.skewed:
            steps.append(("power", PandasPowerTransformer(self.skewed, method="yeo-johnson")))
        steps.append(("scale", PandasRobustScaler(feature_cols)))
        self.stateful = Pipeline(steps).fit(dated)
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """学習済みの値で変換する（各行の値だけから計算するので、行どうしは影響し合わない）。"""
        frame = frame.copy()
        if TARGET_COL in frame.columns:
            # 評価・出力用に元の値を残してから変換する
            position = frame.columns.get_loc(TARGET_COL) + 1
            frame.insert(position, f"{TARGET_COL}_raw", frame[TARGET_COL])
            if self.target_transformer is not None:
                frame = self.target_transformer.transform(frame)
        return self.stateful.transform(self._date_features(frame))

    def inverse_target(self, values: np.ndarray) -> np.ndarray:
        """モデルの尺度の目的変数を、元の尺度に戻す。"""
        if self.target_transformer is None:
            return values
        restored = self.target_transformer.inverse_transform(pd.DataFrame({TARGET_COL: values}))
        return restored[TARGET_COL].to_numpy(dtype=float)

    @staticmethod
    def _date_features(frame: pd.DataFrame) -> pd.DataFrame:
        """日付の成分と周期特徴量を加える（各行の日付だけから作る。学習は不要）。"""
        frame = DatetimeFeaturesExtractor(DATE_COL, components=DATE_COMPONENTS).fit_transform(frame)
        if "month" in DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{DATE_COL}_month", period=12).fit_transform(frame)
        if "weekday" in DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{DATE_COL}_weekday", period=7).fit_transform(frame)
        return frame


def build_train_features(prep: Preprocessor, train: pd.DataFrame) -> pd.DataFrame:
    """学習データの前処理結果に、目的変数のラグ・移動平均を加える（種類ごとに、過去の実測値だけを使う）。

    各種類の先頭 `builder.lookback` 日（過去の日数が足りない日）は除く。
    目的変数や特徴量が欠損の行は残す
    （行を消すと1行 = 1日の前提が崩れ、再帰予測の履歴にも使えなくなるため）。
    """
    features = builder.build(prep.transform(train))
    # 種類ごとの、日付の順の位置（行は日付の昇順に並んでいる）
    position = features.groupby(CAT_COL, sort=False).cumcount()
    return features[position >= builder.lookback].reset_index(drop=True)

## 3. 前処理の実行と出力

In [6]:
prep = Preprocessor().fit(train_raw)
# 学習データ全体（先頭行も含む）の前処理結果。再帰予測の履歴として使う
train_prep = prep.transform(train_raw)
train_features = build_train_features(prep, train_raw)
test_features = prep.transform(test_raw)

date_feature_cols = [c for c in test_features.columns if c.startswith(f"{DATE_COL}_")]
# 種類ごとのモデルの特徴量（自身のラグ・移動平均 + 日付 + 説明変数。種類の列は使わない）
model_feature_cols = [*target_feature_cols, *date_feature_cols, *feature_cols]
n_dropped_days = (len(train_raw) - len(train_features)) // len(cats)
print(f"学習データ: {len(train_features)}行（各種類の先頭 {n_dropped_days}日はラグが作れず除外）")
n_missing_target = int(train_features[TARGET_COL].isna().sum())
print(
    f"  うち目的変数が欠損の行: {n_missing_target}行"
    "（ラグの元として残し、学習の目的変数には使わない）"
)
print(f"テストデータ: {len(test_features)}行")
print(f"モデルの特徴量（種類ごと）: {len(model_feature_cols)}列")
print(f"  目的変数から（テストは再帰予測で作る）: {target_feature_cols}")
print(f"  日付から: {date_feature_cols}")
print(f"  説明変数: {len(feature_cols)}列（うちべき変換 {len(prep.skewed)}列: {prep.skewed}）")

# 出力（目的変数の列は先頭に置く。Y・Y_raw はモデルの特徴量に入れないこと）
target_cols_out = [TARGET_COL, f"{TARGET_COL}_raw"]
train_columns = [
    DATE_COL,
    CAT_COL,
    *target_cols_out,
    *target_feature_cols,
    *date_feature_cols,
    *feature_cols,
]
test_columns = [DATE_COL, CAT_COL, *date_feature_cols, *feature_cols]
assert not set(target_cols_out) & set(model_feature_cols), "目的変数が特徴量に入っています"
assert not set(target_feature_cols) & set(test_columns), "テストに目的変数の特徴量が入っています"
for filename, frame in (
    ("x50_train_features.csv", train_features[train_columns]),
    ("x50_test_features.csv", test_features[test_columns]),
):
    path = ensure_parent_dir(output_dir / filename)
    frame.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"保存: {shown(path)}（{len(frame)}行 × {frame.shape[1]}列）")
train_features[train_columns].head()

学習データ: 10680行（各種類の先頭 28日はラグが作れず除外）
  うち目的変数が欠損の行: 586行（ラグの元として残し、学習の目的変数には使わない）
テストデータ: 3650行
モデルの特徴量（種類ごと）: 69列
  目的変数から（テストは再帰予測で作る）: ['Y_lag_1', 'Y_lag_2', 'Y_lag_3', 'Y_lag_7', 'Y_lag_14', 'Y_lag_28', 'Y_lag_1_ma_7', 'Y_lag_1_ma_28', 'Y_lag_1_roc_1']
  日付から: ['date_year', 'date_month', 'date_day', 'date_weekday', 'date_week', 'date_month_sin', 'date_month_cos', 'date_weekday_sin', 'date_weekday_cos', 'date_elapsed_days']
  説明変数: 50列（うちべき変換 10列: ['X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20']）


保存: data/processed/x50/x50_train_features.csv（10680行 × 73列）


保存: data/processed/x50/x50_test_features.csv（3650行 × 62列）


,date,Y_cat,Y,Y_raw,Y_lag_1,Y_lag_2,Y_lag_3,Y_lag_7,Y_lag_14,Y_lag_28,Y_lag_1_ma_7,Y_lag_1_ma_28,Y_lag_1_roc_1,date_year,date_month,...,X36,X37,X38,X39,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49,X50
0,2022-01-29,Y1,4.206414,67.115441,4.110219,4.006572,4.057716,4.286206,NaN,4.072636,4.157686,4.193086,0.025869,2022,1,...,-0.5,0.666667,-1.0,0.0,0.5,0.394376,0.344019,-0.018021,-1.213705,1.273563,0.195988,-0.850737,0.170235,0.012806,0.33196
1,2022-01-29,Y2,4.525475,92.339737,4.716215,4.488726,4.638476,4.549119,4.416312,4.111354,4.604227,4.539502,0.050680,2022,1,...,-0.5,0.666667,-1.0,0.0,0.5,0.394376,0.344019,-0.018021,-1.213705,1.273563,0.195988,-0.850737,0.170235,0.012806,0.33196
2,2022-01-29,Y3,4.263633,71.067679,4.308905,4.269987,4.151916,NaN,4.011016,4.169372,4.142466,4.092374,0.009114,2022,1,...,-0.5,0.666667,-1.0,0.0,0.5,0.394376,0.344019,-0.018021,-1.213705,1.273563,0.195988,-0.850737,0.170235,0.012806,0.33196
3,2022-01-29,Y4,3.352826,28.583394,3.423561,3.179733,3.374383,3.311253,3.325183,3.370503,3.351396,3.398366,0.076682,2022,1,...,-0.5,0.666667,-1.0,0.0,0.5,0.394376,0.344019,-0.018021,-1.213705,1.273563,0.195988,-0.850737,0.170235,0.012806,0.33196
4,2022-01-29,Y5,4.163728,64.310797,4.137266,4.177821,4.188218,4.280470,4.404264,4.325433,4.236234,4.292219,-0.009707,2022,1,...,-0.5,0.666667,-1.0,0.0,0.5,0.394376,0.344019,-0.018021,-1.213705,1.273563,0.195988,-0.850737,0.170235,0.012806,0.33196


## 4. 再帰予測

種類（`Y_cat`）ごとに、その種類の学習データの行（自身のラグ・移動平均を含む）で1日先を予測するモデルを学習する
（目的変数が欠損の行は学習に使わない）。
テスト期間は `modeling.forecasting.recursive_forecast` で1日ずつ進む。各日について:

1. 学習期間の実測値と、前日までの予測値をつないだ履歴から、その日の目的変数のラグ・移動平均・前日比を作る
   （`TargetFeatureBuilder`。学習データと同じ定義。学習期間の末尾の欠損は欠損のまま使う）。
2. その日の日付・説明変数の特徴量（`x50_test_features.csv` の値）と合わせて予測する。
3. 予測値を履歴に足し、翌日に進む。

予測値は対数の尺度なので、最後に元の尺度に戻す。

In [7]:
models: dict[str, LGBMRegressor] = {}
for cat in cats:
    # 目的変数が欠損の行は学習に使わない（特徴量の欠損は LightGBM がそのまま扱う）
    rows = (train_features[CAT_COL] == cat) & train_features[TARGET_COL].notna()
    models[cat] = LGBMRegressor(**MODEL_PARAMS, verbose=-1).fit(
        train_features.loc[rows, model_feature_cols], train_features.loc[rows, TARGET_COL]
    )


def forecast(cat: str, test_features: pd.DataFrame) -> np.ndarray:
    """種類 cat のテスト期間を再帰予測し、モデルの尺度（変換後）の予測値を返す（日付の順）。"""
    return recursive_forecast(
        models[cat],
        train_prep[train_prep[CAT_COL] == cat],
        test_features[test_features[CAT_COL] == cat],
        builder,
        model_feature_cols,
        apply_clip=False,
    )


prediction_parts = []
pred_model: dict[str, np.ndarray] = {}
for cat in cats:
    pred_model[cat] = forecast(cat, test_features)
    future = test_features.loc[test_features[CAT_COL] == cat, [DATE_COL, CAT_COL]]
    history_cat = train_prep.loc[train_prep[CAT_COL] == cat, [DATE_COL, CAT_COL, TARGET_COL]]

    # 予測に使った目的変数の特徴量を、履歴（実測値 + 予測値）から作り直して記録する
    history = pd.concat(
        [history_cat, future.assign(**{TARGET_COL: pred_model[cat]})], ignore_index=True
    )
    recursion = builder.build(history).iloc[len(history_cat) :].reset_index(drop=True)
    # 作り直した特徴量で予測すると再帰予測の値と一致する（＝この特徴量を使って予測した）
    replay_input = pd.concat(
        [
            recursion[target_feature_cols],
            test_features[test_features[CAT_COL] == cat].reset_index(drop=True),
        ],
        axis=1,
    )
    replayed = models[cat].predict(replay_input[model_feature_cols])
    np.testing.assert_allclose(replayed, pred_model[cat], rtol=1e-9, atol=1e-9)

    prediction_parts.append(
        pd.concat(
            [
                future.reset_index(drop=True),
                pd.DataFrame(
                    {
                        f"{TARGET_COL}_pred": prep.inverse_target(pred_model[cat]),
                        f"{TARGET_COL}_pred_model": pred_model[cat],
                    }
                ),
                recursion[target_feature_cols],
            ],
            axis=1,
        )
    )

predictions = sort_long(pd.concat(prediction_parts, ignore_index=True), cats)
path = ensure_parent_dir(output_dir / "x50_test_predictions.csv")
predictions.to_csv(path, index=False, date_format="%Y-%m-%d")
print(f"保存: {shown(path)}（{len(predictions)}行 × {predictions.shape[1]}列）")
predictions.head()

保存: data/processed/x50/x50_test_predictions.csv（3650行 × 13列）


,date,Y_cat,Y_pred,Y_pred_model,Y_lag_1,Y_lag_2,Y_lag_3,Y_lag_7,Y_lag_14,Y_lag_28,Y_lag_1_ma_7,Y_lag_1_ma_28,Y_lag_1_roc_1
0,2025-01-01,Y1,80.846914,4.392557,4.505257,4.500699,4.529936,4.426126,4.415400,4.313067,4.472458,4.390490,0.001013
1,2025-01-01,Y2,66.565399,4.198185,4.141091,4.246141,4.427030,4.128424,4.241995,4.083691,4.215433,4.247146,-0.024740
2,2025-01-01,Y3,80.073830,4.382949,4.238918,4.277843,4.198623,4.251439,4.288278,4.311335,4.237966,4.283958,-0.009099
3,2025-01-01,Y4,33.635843,3.515592,3.605440,3.511194,3.527014,3.506190,3.445289,3.511897,3.548350,3.547373,0.026841
4,2025-01-01,Y5,97.266938,4.577459,4.620947,4.636428,4.670971,4.811279,4.659399,4.550993,4.704140,4.585349,-0.003339


## 5. 予測の評価（正解ファイルがある場合）

正解はここで初めて読む（前処理・学習・予測には使っていない）。種類ごとに MAE を計算し、比較の基準として
学習データの最終日の値をそのまま使い続ける予測（ナイーブ予測）の MAE も示す。種類ごとに尺度が違うため、
MAE を平均の水準で割った値（相対MAE）も載せる。正解が欠損の日は評価から除く。ナイーブ予測には、学習データの
最後の観測値（欠損でない値）を使う。

In [8]:
if answer_path is not None:
    answer = read_csv_auto(answer_path)
    merged = predictions[[DATE_COL, CAT_COL, f"{TARGET_COL}_pred"]].merge(
        answer[[DATE_COL, CAT_COL, TARGET_COL]],
        on=[DATE_COL, CAT_COL],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    # 正解ファイルに全ての日付・種類の行があること（値の欠損は評価から除く）
    assert len(merged) == len(predictions) and (merged["_merge"] == "both").all()
    merged = merged.drop(columns="_merge")
    # 各種類の学習データの最後の観測値（groupby の last は欠損を飛ばす）
    last_train = train_raw.groupby(CAT_COL)[TARGET_COL].last()
    rows = []
    for cat in cats:
        part = merged[merged[CAT_COL] == cat]
        # 正解が欠損の日の誤差は欠損になり、mean() で除かれる
        error = part[f"{TARGET_COL}_pred"] - part[TARGET_COL]
        naive_error = last_train[cat] - part[TARGET_COL]
        rows.append(
            {
                CAT_COL: cat,
                "MAE（再帰予測）": error.abs().mean(),
                "MAE（ナイーブ）": naive_error.abs().mean(),
                "相対MAE（再帰予測）": error.abs().mean() / part[TARGET_COL].mean(),
                "評価した日数": int(part[TARGET_COL].notna().sum()),
            }
        )
    evaluation = pd.DataFrame(rows)
    period = f"{merged[DATE_COL].min():%Y-%m-%d}〜{merged[DATE_COL].max():%Y-%m-%d}"
    n_missing_answer = int(merged[TARGET_COL].isna().sum())
    print(
        f"評価: {merged[DATE_COL].nunique()}日 × {len(cats)}種類（{period}）。"
        f"正解が欠損の {n_missing_answer}行は除く"
    )
    better = int((evaluation["MAE（再帰予測）"] < evaluation["MAE（ナイーブ）"]).sum())
    print(f"再帰予測がナイーブ予測より良い種類: {better} / {len(cats)}")
    display(evaluation.round(3))

評価: 365日 × 10種類（2025-01-01〜2025-12-31）。正解が欠損の 238行は除く
再帰予測がナイーブ予測より良い種類: 10 / 10


,Y_cat,MAE（再帰予測）,MAE（ナイーブ）,相対MAE（再帰予測）,評価した日数
0,Y1,3.452,19.376,0.048,345
1,Y2,4.603,12.131,0.064,349
2,Y3,17.346,18.481,0.199,340
3,Y4,1.235,5.853,0.040,352
4,Y5,6.682,19.281,0.080,323
5,Y6,11.910,30.121,0.066,349
6,Y7,7.892,17.511,0.040,353
7,Y8,10.484,62.484,0.053,328
8,Y9,10.819,18.264,0.064,328
9,Y10,1.713,7.404,0.051,345


## 6. リークの確認

入力を書き換えて再実行し、未来の値や他の行・他の種類の値を使っていないことを確かめる。

1. **テストデータは学習側の結果に影響しない:** テストデータの説明変数を乱数に置き換えても、
   学習データの前処理結果は変わらない（前処理は学習データだけで fit している）。
2. **テストデータの前処理は行ごと:** テストデータのある1日の説明変数だけを書き換えると、変わるのはその日の行だけ。
3. **再帰予測は未来を見ない:** テスト期間のある日 t0 以降の説明変数を書き換えても、どの種類でも
   t0 より前の予測値は変わらない。
4. **学習データのラグは当日の値・他の種類を含まない:** 学習データのある日 t0（Y1 の `Y` が欠損でない日）の
   Y1 の `Y` だけを書き換えると、特徴量が変わるのは Y1 の行の t0 の翌日以降だけ（他の種類の行の特徴量は変わらない）。

In [9]:
rng = np.random.default_rng(0)


def with_random(frame: pd.DataFrame, mask: pd.Series, columns: list[str]) -> pd.DataFrame:
    """mask の行の指定列を、元とかけ離れた正の乱数に置き換えた表を返す。"""
    changed = frame.copy()
    changed.loc[mask, columns] = rng.uniform(1e3, 1e4, size=(int(mask.sum()), len(columns)))
    return changed


# 1. 前処理の fit にはテストデータを渡していないので、学習データの結果はテストデータに依存しない。
#    念のため、前処理をやり直しても学習データの結果が同じで、テストの変更が反映されることを確かめる
all_rows = pd.Series(True, index=test_raw.index)
changed_test = with_random(test_raw, all_rows, feature_cols)
pd.testing.assert_frame_equal(
    build_train_features(Preprocessor().fit(train_raw), train_raw), train_features
)
assert not prep.transform(changed_test).equals(test_features)
print("OK 1: テストデータは学習データの前処理結果に影響しない")

# 2. テストデータの1日だけ（全種類の行）を書き換えると、変わるのはその日の行だけ
t0 = test_raw[DATE_COL].drop_duplicates().iloc[100]
one_day = test_raw[DATE_COL] == t0
changed = prep.transform(with_random(test_raw, one_day, feature_cols))[test_columns]
same = changed.eq(test_features[test_columns]) | (
    changed.isna() & test_features[test_columns].isna()
)
diff_rows = ~same.all(axis=1)
assert diff_rows.equals(one_day), diff_rows[diff_rows].index.tolist()
print(f"OK 2: {t0:%Y-%m-%d} の説明変数を書き換えると、その日の行だけが変わる")

# 3. t0 以降の説明変数を書き換えても、t0 より前の予測値は同じ（全種類）
changed_after = prep.transform(with_random(test_raw, test_raw[DATE_COL] >= t0, feature_cols))
for cat in cats:
    after = (test_features.loc[test_features[CAT_COL] == cat, DATE_COL] >= t0).to_numpy()
    pred_changed = forecast(cat, changed_after)
    np.testing.assert_array_equal(pred_changed[~after], pred_model[cat][~after])
    assert not np.allclose(pred_changed[after], pred_model[cat][after]), cat
n_before = int((test_raw[DATE_COL].drop_duplicates() < t0).sum())
print(f"OK 3: {t0:%Y-%m-%d} 以降を書き換えても、全種類でそれより前の {n_before}日の予測値は同じ")

# 4. 学習データの t0 の Y1 の値だけを書き換えると、変わるのは Y1 の行の翌日以降の特徴量だけ
changed_cat = cats[0]
# 書き換える日は、その種類の目的変数が欠損でない日にする（欠損の値は10倍しても欠損のまま）
observed = train_raw.loc[
    (train_raw[CAT_COL] == changed_cat) & train_raw[TARGET_COL].notna(), DATE_COL
]
t0_train = observed[observed >= train_raw[DATE_COL].drop_duplicates().iloc[500]].min()
changed_train = train_raw.copy()
target_row = (changed_train[DATE_COL] == t0_train) & (changed_train[CAT_COL] == changed_cat)
changed_train.loc[target_row, TARGET_COL] *= 10
rebuilt = build_train_features(prep, changed_train)
# 欠損どうしは同じとみなして比べる（NaN != NaN は True になるため）
before, after_change = train_features[target_feature_cols], rebuilt[target_feature_cols]
diff = ~(after_change.eq(before) | (after_change.isna() & before.isna())).all(axis=1)
assert set(rebuilt.loc[diff, CAT_COL]) == {changed_cat}, "他の種類の特徴量が変わりました"
first_changed = rebuilt.loc[diff, DATE_COL].min()
assert first_changed == t0_train + pd.Timedelta(days=1), first_changed
print(
    f"OK 4: 学習データの {t0_train:%Y-%m-%d} の {changed_cat} は、{changed_cat} の"
    f"翌日 {first_changed:%Y-%m-%d} 以降の特徴量にだけ使われる（他の種類の特徴量は不変）"
)

OK 1: テストデータは学習データの前処理結果に影響しない
OK 2: 2025-04-11 の説明変数を書き換えると、その日の行だけが変わる


OK 3: 2025-04-11 以降を書き換えても、全種類でそれより前の 100日の予測値は同じ


OK 4: 学習データの 2023-05-16 の Y1 は、Y1 の翌日 2023-05-17 以降の特徴量にだけ使われる（他の種類の特徴量は不変）


## まとめ

- long 型（1行 = 日付 × 目的変数の種類）の学習データ・テストデータのそれぞれに、`src/feature_engineering` の前処理
  （目的変数の対数変換、日付の成分・周期・経過日数、説明変数のべき変換・スケーリング）をかけ、`data/processed/x50/` に
  long 型の CSV で出力した。前処理は学習データだけで fit した。
- 目的変数のラグ・移動平均・前日比は、学習データでは種類ごとに自身の過去の実測値から作り、テスト期間では
  種類ごとの再帰予測の中で「実測値 + それまでの予測値」から1日ずつ作った
  （`x50_test_predictions.csv` に、予測に使った値を記録した）。
- 「リークの確認」で、テストデータ・未来の値・当日の目的変数・他の種類の値が前処理や予測に使われていないことを確かめた。

**注意点**

- 再帰予測では、先の日ほど「予測値から作ったラグ」に頼るため、誤差が積み重なる。
- 種類ごとに尺度が違うため、MAE は種類ごとに比べる（全体を比べるときは相対MAE を使う）。
- 日付の特徴量 `date_year` は、テスト期間（2025年）の値が学習データに無い。木のモデルは学習範囲の外の値を
  学習範囲の端と同じに扱う。トレンドのある種類（ダミーでは Y3・Y6・Y9）は、学習範囲より先の水準を当てにくい。
- 欠損は埋めていない。目的変数が欠損の行は学習・評価から除き、ラグの元としては欠損のまま残した
  （移動平均は窓内の観測値だけで平均）。説明変数・目的変数の特徴量の欠損は LightGBM の欠損の扱いに任せた。
  長い連続した欠損の直後は、ラグ・移動平均が欠損または少ない観測値から作られるため、予測が不安定になりやすい。